In [1]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.linear_model import LogisticRegression
import statsmodels.api as sm

import pickle

import xgboost as xgb
print(xgb.__version__)

3.2.0


In [2]:
train = pd.read_csv('../Data/train_features.csv')
oot1 = pd.read_csv('../Data/oot1_features.csv')
oot2 = pd.read_csv('../Data/oot2_features.csv')

all_features = pickle.load(open('../Files/all_features.pkl', 'rb'))
identifier_cols = all_features['identifier_features']
features_to_select = pickle.load(open('../Files/selected_features.csv', 'rb'))['final_selected_features']

In [3]:
identifier_cols

['id', 'stmt_date', 'data_type', 'default', 'ateco_sector']

In [4]:
imputer = pd.read_csv('../Data/feature_imputer.csv')

In [5]:
cols_to_impute = list(set(imputer.columns) - set(['ateco_sector', 'log_assets']))
imputed_flag_cols = [f"{col}_imputed_flag" for col in cols_to_impute]
cols_to_impute

['cash_to_debt_st',
 'debt_to_ebitda',
 'interest_coverage_proxy',
 'operating_margin',
 'operating_cashflow_to_liabilities',
 'current_assets_to_debt_st',
 'cash_flow_to_debt']

In [6]:
def missing_value_imputation(df, imputation_table, cols_to_impute, asset_col='asst_tot'):
    df_copy = df.copy()
    for c in df.columns:
        if c in cols_to_impute: df[c] = pd.to_numeric(df[c], errors="coerce")

    # Create missingness flags before imputation
    flag_cols = {col: f"{col}_imputed_flag" for col in cols_to_impute}
    for col, flag_col in flag_cols.items(): df_copy[flag_col] = df_copy[col].isna().astype(int)

    imputation_table = imputation_table.set_index("ateco_sector")

    row_medians = imputation_table.reindex(df_copy["ateco_sector"]).set_axis(df_copy.index)
    row_medians = row_medians.fillna(imputation_table.loc["global"])
    assets = df_copy[asset_col].fillna(row_medians[asset_col])

    df_copy[cols_to_impute] = df_copy[cols_to_impute].fillna(row_medians[cols_to_impute].mul(assets, axis=0))

    return df_copy

In [7]:
train_imp = missing_value_imputation(train, imputer, cols_to_impute, asset_col='log_assets')
oot1_imp = missing_value_imputation(oot1, imputer, cols_to_impute, asset_col='log_assets')
oot2_imp = missing_value_imputation(oot2, imputer, cols_to_impute, asset_col='log_assets')

In [12]:
vif = pd.read_csv('../Files/VIF.csv').sort_values(by='VIF', ascending=False)
features_to_select = vif.query('feature != "const"')['feature'].tolist()
vif

,feature,VIF
8,operating_loss_flag,3.297599
9,nonpositive_ebitda_flag,3.039712
18,debt_st_to_assets,2.877178
17,equity_to_assets,1.812351
13,working_capital_to_assets,1.713624
7,net_loss_flag,1.552789
6,receivables_to_assets,1.491508
1,operating_cashflow_to_liabilities,1.385908
11,cash_to_debt_st,1.332261
12,current_assets_to_debt_st,1.272679


In [14]:
train_imp = train_imp[identifier_cols + features_to_select]
oot1_imp = oot1_imp[identifier_cols + features_to_select]
oot2_imp = oot2_imp[identifier_cols + features_to_select]

In [25]:
X = train_imp.copy()
y = pd.Series(np.asarray(train_imp['default']), index=X.index)

flag_cols = [c for c in features_to_select if 'flag' in c]

rows = []

for col in flag_cols:
    table = pd.crosstab(X[col], y).reindex(
        index=[0, 1],
        columns=[0, 1],
        fill_value=0
    )

    rows.append({
        "feature": col,
        "n_flagged": table.loc[1].sum(),
        "defaults_flagged": table.loc[1, 1],
        "nondefaults_flagged": table.loc[1, 0],
        "default_rate_flagged": (
            table.loc[1, 1] / table.loc[1].sum()
            if table.loc[1].sum() > 0 else np.nan
        ),
        "zero_cell": (table == 0).any().any()
    })

flag_summary = pd.DataFrame(rows)

display(flag_summary.sort_values(
    ["zero_cell", "n_flagged"],
    ascending=[False, True]
))

,feature,n_flagged,defaults_flagged,nondefaults_flagged,default_rate_flagged,zero_cell
6,operating_cashflow_to_liabilities_imputed_flag,93,0,93,0.000000,True
5,cash_to_debt_st_imputed_flag,952,15,937,0.015756,False
7,interest_coverage_proxy_imputed_flag,15469,180,15289,0.011636,False
3,operating_margin_imputed_flag,15583,316,15267,0.020279,False
4,negative_equity_flag,17744,1631,16113,0.091918,False
1,nonpositive_ebitda_flag,73229,2962,70267,0.040448,False
0,operating_loss_flag,104076,3447,100629,0.033120,False
2,net_loss_flag,164335,4329,160006,0.026343,False


In [16]:
X = sm.add_constant(train_imp[features_to_select].astype(float), has_constant='add')
y = np.asarray(train['default'], dtype=float)

In [17]:
result = sm.Logit(y, X).fit(maxiter=5000)
ci = result.conf_int(alpha=0.05)

         Current function value: 0.064441
         Iterations: 5000


/opt/homebrew/Caskroom/miniconda/base/envs/nlp/lib/python3.10/site-packages/statsmodels/discrete/discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


In [22]:
print(result.summary())

                           Logit Regression Results                           
Dep. Variable:                      y   No. Observations:               475389
Model:                          Logit   Df Residuals:                   475366
Method:                           MLE   Df Model:                           22
Date:                Sun, 15 Nov 2026   Pseudo R-squ.:                  0.1246
Time:                        22:11:25   Log-Likelihood:                -30634.
converged:                      False   LL-Null:                       -34996.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                                     coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------------------
const                                             -6.2467      0.055   -113.557      0.000      -6.354      -6.139
operating_loss_flag    

In [23]:
print(result.summary2())

                                              Results: Logit
Model:                             Logit                         Method:                        MLE       
Dependent Variable:                y                             Pseudo R-squared:              0.125     
Date:                              2026-11-15 22:11              AIC:                           61314.7782
No. Observations:                  475389                        BIC:                           61569.4317
Df Model:                          22                            Log-Likelihood:                -30634.   
Df Residuals:                      475366                        LL-Null:                       -34996.   
Converged:                         0.0000                        LLR p-value:                   0.0000    
No. Iterations:                    5000.0000                     Scale:                         1.0000    
-----------------------------------------------------------------------------------

In [24]:
print(result.get_margeff().summary())

        Logit Marginal Effects       
Dep. Variable:                      y
Method:                          dydx
At:                           overall
                                                    dy/dx    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------------------
operating_loss_flag                                0.0038      0.001      5.495      0.000       0.002       0.005
nonpositive_ebitda_flag                            0.0080      0.001     11.872      0.000       0.007       0.009
debt_st_to_assets                                  0.0191      0.001     20.331      0.000       0.017       0.021
equity_to_assets                                   0.0034      0.001      4.393      0.000       0.002       0.005
working_capital_to_assets                          0.0090      0.001     13.468      0.000       0.008       0.010
net_loss_flag                              

In [30]:
summary = pd.DataFrame({
    'beta': result.params,
    'std_error': result.bse,
    'p_value': result.pvalues,
    'ci_lower': ci[0],
    'ci_upper': ci[1],
    'odds_ratio': np.exp(result.params),
    'or_ci_lower': np.exp(ci[0]),
    'or_ci_upper': np.exp(ci[1])
})
summary.to_csv('../Artifacts/logit_model_summary.csv')
display(summary.sort_values(by='p_value', ascending=False))

/opt/homebrew/Caskroom/miniconda/base/envs/nlp/lib/python3.10/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: overflow encountered in exp
  result = getattr(ufunc, method)(*inputs, **kwargs)


,beta,std_error,p_value,ci_lower,ci_upper,odds_ratio,or_ci_lower,or_ci_upper
operating_cashflow_to_liabilities_imputed_flag,-1.893111e+01,1.023753e+04,9.985246e-01,-2.008412e+04,20046.254116,6.002364e-09,0.000000,inf
cash_flow_to_debt,1.890246e-05,7.501309e-04,9.798963e-01,-1.451327e-03,0.001489,1.000019e+00,0.998550,1.001490
ebitda_to_assets,-3.154083e-04,8.008912e-03,9.685857e-01,-1.601259e-02,0.015382,9.996846e-01,0.984115,1.015501
operating_cashflow_to_assets,-2.125840e-02,5.825721e-02,7.151816e-01,-1.354404e-01,0.092924,9.789660e-01,0.873331,1.097378
debt_to_ebitda,-4.821663e-06,1.039166e-05,6.426524e-01,-2.518894e-05,0.000016,9.999952e-01,0.999975,1.000016
operating_cashflow_to_liabilities,4.863068e-04,9.025378e-04,5.900100e-01,-1.282635e-03,0.002255,1.000486e+00,0.998718,1.002258
interest_coverage_proxy,4.034997e-07,3.061126e-07,1.874563e-01,-1.964700e-07,0.000001,1.000000e+00,1.000000,1.000001
cash_to_debt_st,2.207404e-04,1.424000e-04,1.211073e-01,-5.835859e-05,0.000500,1.000221e+00,0.999942,1.000500
cash_to_debt_st_imputed_flag,4.333326e-01,2.685251e-01,1.065815e-01,-9.296689e-02,0.959632,1.542389e+00,0.911224,2.610736
current_assets_to_debt_st,-2.260381e-04,1.336435e-04,9.076965e-02,-4.879745e-04,0.000036,9.997740e-01,0.999512,1.000036


In [29]:
pickle.dump(result, open('../Artifacts/logit_model.pkl', 'wb'))

In [36]:
X_train_sm = sm.add_constant(train_imp[features_to_select].astype(float), has_constant='add')
X_train_sm = X_train_sm[result.model.exog_names]
train_pred = result.predict(X_train_sm)

X_oot1_sm = sm.add_constant(oot1_imp[features_to_select].astype(float), has_constant='add')
X_oot1_sm = X_oot1_sm[result.model.exog_names]
oot1_pred = result.predict(X_oot1_sm)

X_oot2_sm = sm.add_constant(oot2_imp[features_to_select].dropna().astype(float), has_constant='add')
X_oot2_sm = X_oot2_sm[result.model.exog_names]
oot2_pred = result.predict(X_oot2_sm)

print(f'Train AUC: {roc_auc_score(train_imp["default"], train_pred):.4f}')
print(f'OOT1 AUC: {roc_auc_score(oot1_imp["default"], oot1_pred):.4f}')
print(f'OOT2 AUC: {roc_auc_score(oot2_imp["default"], oot2_pred):.4f}')

/opt/homebrew/Caskroom/miniconda/base/envs/nlp/lib/python3.10/site-packages/statsmodels/discrete/discrete_model.py:2620: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-X))


Train AUC: 0.7717
OOT1 AUC: 0.7606
OOT2 AUC: 0.7820


In [37]:
scores_train = pd.concat([ID_train, pd.DataFrame({'y_true': y_train, 'y_pred': train_pred})], axis=1)
scores_oot1 = pd.concat([ID_oot1, pd.DataFrame({'y_true': y_oot1, 'y_pred': oot1_pred})], axis=1)
scores_oot2 = pd.concat([ID_oot2.loc[X_oot2[vars_without_cat].dropna().index], pd.DataFrame({'y_true': y_oot2.loc[X_oot2[vars_without_cat].dropna().index], 'y_pred': oot2_pred})], axis=1)

NameError: name 'ID_train' is not defined

In [ ]:
scores_train.to_csv('../Model_scores/scores_train_imputed_logit.csv', index=False)
scores_oot1.to_csv('../Model_scores/scores_oot1_imputed_logit.csv', index=False)
scores_oot2.to_csv('../Model_scores/scores_oot2_imputed_logit.csv', index=False)

# features[id_vars + categorical_features].to_csv('../Model_scores/categorical_features.csv', index=False)

In [ ]:
roc_auc_train = roc_auc_score(scores_train['y_true'], scores_train['y_pred'])
roc_auc_oot1 = roc_auc_score(scores_oot1['y_true'], scores_oot1['y_pred'])
roc_auc_oot2 = roc_auc_score(scores_oot2['y_true'], scores_oot2['y_pred'])

print(f"Train ROC AUC: {roc_auc_train:.4f}")
print(f"oot1 ROC AUC: {roc_auc_oot1:.4f}")
print(f"OOT2 ROC AUC: {roc_auc_oot2:.4f}")

In [ ]:
fpr_train, tpr_train, thresholds_train = roc_curve(scores_train['y_true'], scores_train['y_pred'])
fpr_oot1, tpr_oot1, thresholds_oot1 = roc_curve(scores_oot1['y_true'], scores_oot1['y_pred'])
fpr_oot2, tpr_oot2, thresholds_oot2 = roc_curve(scores_oot2['y_true'], scores_oot2['y_pred'])

In [ ]:
plt.plot(fpr_train, tpr_train, label='Train ROC curve (area = %0.2f)' % roc_auc_score(scores_train['y_true'], scores_train['y_pred']))
plt.plot(fpr_oot1, tpr_oot1, label='OOT1 ROC curve (area = %0.2f)' % roc_auc_score(scores_oot1['y_true'], scores_oot1['y_pred']))
plt.plot(fpr_oot2, tpr_oot2, label='OOT2 ROC curve (area = %0.2f)' % roc_auc_score(scores_oot2['y_true'], scores_oot2['y_pred']))
plt.legend()
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves')
plt.show()

In [ ]:
y = scores_oot2['y_true'].to_numpy()
score = scores_oot2['y_pred'].to_numpy()

n_bootstrap = 100
rng = np.random.default_rng(42)
aucs = []

plt.figure(figsize=(8, 6))

for i in tqdm(range(n_bootstrap)):
    indices = rng.choice(len(y), size=len(y), replace=True)
    y_bootstrap = y[indices]
    score_bootstrap = score[indices]

    roc_auc_bootstrap = roc_auc_score(y_bootstrap, score_bootstrap)
    aucs.append(roc_auc_bootstrap)
    fpr_bootstrap, tpr_bootstrap, _ = roc_curve(y_bootstrap, score_bootstrap)

    plt.plot(fpr_bootstrap, tpr_bootstrap, 'b', alpha=0.6)

auc_mean = np.mean(aucs)
auc_lower, auc_upper = np.percentile(aucs, [2.5, 97.5])
print(f"Mean bootstrap AUC: {auc_mean:.4f}")
print(f"95% confidence interval: [{auc_lower:.4f}, {auc_upper:.4f}]")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend(fontsize="small")
plt.grid(alpha=0.5)
plt.show()